In [19]:
import requests
import json
import time
import math
from datetime import datetime, timedelta
import pandas as pd

In [20]:
AERODATABOX_KEY = "a70a952394msh6b78ca787f2c543p19188cjsn507f0843caaa"
AERODATABOX_HOST = "aerodatabox.p.rapidapi.com"
HEADERS = {"x-rapidapi-host": AERODATABOX_HOST, "x-rapidapi-key": AERODATABOX_KEY}
AIRPORT = "LHR"
RATE_LIMIT_PAUSE = 1.2


def _fetch_window(from_local: str, to_local: str) -> dict:
    """Fetch one ≤12h window from the API."""
    url = (
        f"https://{AERODATABOX_HOST}/flights/airports/iata/{AIRPORT}"
        f"/{from_local}/{to_local}"
        f"?direction=Both&withCancelled=true&withCodeshared=true"
        f"&withLocation=false&withCargo=true"
    )
    resp = requests.get(url, headers=HEADERS, timeout=30)
    if resp.status_code == 204:
        return {"arrivals": [], "departures": []}
    resp.raise_for_status()
    return resp.json()


def _fetch_range(from_dt: datetime, to_dt: datetime) -> dict:
    """Fetch an arbitrary range by chunking into ≤12h windows."""
    all_arrivals = []
    all_departures = []

    current = from_dt
    while current < to_dt:
        chunk_end = min(current + timedelta(hours=12), to_dt)
        from_str = current.strftime("%Y-%m-%dT%H:%M")
        to_str = chunk_end.strftime("%Y-%m-%dT%H:%M")

        print(f"  Fetching {from_str} → {to_str}")
        data = _fetch_window(from_str, to_str)
        all_arrivals.extend(data.get("arrivals", []))
        all_departures.extend(data.get("departures", []))

        current = chunk_end
        if current < to_dt:
            time.sleep(RATE_LIMIT_PAUSE)

    return {"arrivals": all_arrivals, "departures": all_departures}


def count_flights(from_dt: str, to_dt: str, airport: str = AIRPORT) -> dict:
    """
    Count deduplicated, non-cancelled flights in a datetime range.

    Args:
        from_dt: Start datetime inclusive, local time (e.g. "2026-02-22 00:00")
        to_dt:   End datetime exclusive, local time (e.g. "2026-03-01 00:00")
        airport: IATA code (default LHR)

    Returns:
        dict with landings, takeoffs, total, and api_calls_used
    """
    start = datetime.strptime(from_dt, "%Y-%m-%d %H:%M")
    end = datetime.strptime(to_dt, "%Y-%m-%d %H:%M")
    calls_needed = math.ceil((end - start).total_seconds() / (12 * 3600))
    print(f"Range: {from_dt} → {to_dt} ({calls_needed} API calls needed)")

    data = _fetch_range(start, end)

    # ── Remove cancellations ──────────────────────────────────────────────
    def is_not_cancelled(f):
        status = (f.get("status") or "").lower()
        return not status.startswith("cancel")

    arrivals = [f for f in data["arrivals"] if is_not_cancelled(f)]
    departures = [f for f in data["departures"] if is_not_cancelled(f)]

    # ── Deduplicate codeshares ────────────────────────────────────────────
    def dedup(flights):
        groups = {}
        for f in flights:
            movement = f.get("movement", {}) or {}
            scheduled = (movement.get("scheduledTime", {}) or {}).get("utc", "")
            other_iata = (movement.get("airport", {}) or {}).get("iata", "")
            key = f"{scheduled}|{other_iata}"

            cs_status = f.get("codeshareStatus", "")
            rank = 0 if cs_status == "IsOperator" else (1 if cs_status == "" else 2)

            if key not in groups or rank < groups[key][0]:
                groups[key] = (rank, f)

        return [v[1] for v in groups.values()]

    arrivals = dedup(arrivals)
    departures = dedup(departures)

    return {
        "from": from_dt,
        "to": to_dt,
        "landings": len(arrivals),
        "takeoffs": len(departures),
        "total": len(arrivals) + len(departures),
        "api_calls_used": calls_needed,
        "arrivals_raw": arrivals,
        "departures_raw": departures
    }

In [21]:
result = count_flights("2026-02-28 12:00", "2026-03-01 12:00")
print(result)

Range: 2026-02-28 12:00 → 2026-03-01 12:00 (2 API calls needed)
  Fetching 2026-02-28T12:00 → 2026-03-01T00:00
  Fetching 2026-03-01T00:00 → 2026-03-01T12:00
{'from': '2026-02-28 12:00', 'to': '2026-03-01 12:00', 'landings': 568, 'takeoffs': 599, 'total': 1167, 'api_calls_used': 2, 'arrivals_raw': [{'movement': {'airport': {'icao': 'KORD', 'iata': 'ORD', 'name': 'Chicago', 'countryCode': 'us', 'timeZone': 'America/Chicago'}, 'scheduledTime': {'utc': '2026-02-28 11:35Z', 'local': '2026-02-28 11:35+00:00'}, 'revisedTime': {'utc': '2026-02-28 12:01Z', 'local': '2026-02-28 12:01+00:00'}, 'runwayTime': {'utc': '2026-02-28 12:01Z', 'local': '2026-02-28 12:01+00:00'}, 'terminal': '5', 'baggageBelt': '3', 'quality': ['Basic', 'Live']}, 'number': 'BA 296', 'callSign': 'BAW6AC', 'status': 'Arrived', 'codeshareStatus': 'IsOperator', 'isCargo': False, 'aircraft': {'reg': 'G-VIIY', 'modeS': '400776', 'model': 'Boeing 777-200 / 200ER Passenger'}, 'airline': {'name': 'British', 'iata': 'BA', 'icao': 

In [22]:
df_arrivals = pd.DataFrame(result['arrivals_raw'])

In [23]:
df_arrivals

,movement,number,callSign,status,codeshareStatus,isCargo,aircraft,airline
0,"{'airport': {'icao': 'KORD', 'iata': 'ORD', 'n...",BA 296,BAW6AC,Arrived,IsOperator,False,"{'reg': 'G-VIIY', 'modeS': '400776', 'model': ...","{'name': 'British', 'iata': 'BA', 'icao': 'BAW'}"
1,"{'airport': {'icao': 'EGPF', 'iata': 'GLA', 'n...",BA 1477,SHT7T,Arrived,IsOperator,False,"{'reg': 'G-TTSF', 'modeS': '4081BF', 'model': ...","{'name': 'British', 'iata': 'BA', 'icao': 'BAW'}"
2,"{'airport': {'icao': 'EIDW', 'iata': 'DUB', 'n...",BA 829,BAW8ER,Arrived,IsOperator,False,"{'reg': 'G-TTSG', 'modeS': '40822E', 'model': ...","{'name': 'British', 'iata': 'BA', 'icao': 'BAW'}"
3,"{'airport': {'icao': 'LKPR', 'iata': 'PRG', 'n...",BA 853,BAW853S,Arrived,IsOperator,False,"{'reg': 'G-EUUG', 'modeS': '400982', 'model': ...","{'name': 'British', 'iata': 'BA', 'icao': 'BAW'}"
4,"{'airport': {'icao': 'LFML', 'iata': 'MRS', 'n...",BA 365,BAW365,Arrived,IsOperator,False,"{'reg': 'G-EUPU', 'modeS': '4008B5', 'model': ...","{'name': 'British', 'iata': 'BA', 'icao': 'BAW'}"
...,...,...,...,...,...,...,...,...
563,"{'airport': {'icao': 'CYVR', 'iata': 'YVR', 'n...",AC 860,NaN,Expected,IsOperator,False,{'model': 'Boeing 777-300ER Passenger'},"{'name': 'Air Canada', 'iata': 'AC', 'icao': '..."
564,"{'airport': {'icao': 'EHAM', 'iata': 'AMS', 'n...",BA 433,NaN,Expected,IsOperator,False,{'model': 'Airbus A319'},"{'name': 'British', 'iata': 'BA', 'icao': 'BAW'}"
565,"{'airport': {'icao': 'KSEA', 'iata': 'SEA', 'n...",VS 106,VIR106F,Expected,IsOperator,False,"{'reg': 'G-VWOO', 'modeS': '4072EA', 'model': ...","{'name': 'Virgin Atlantic', 'iata': 'VS', 'ica..."
566,"{'airport': {'icao': 'EGPF', 'iata': 'GLA', 'n...",BA 1477,NaN,Expected,IsOperator,False,{'model': 'Airbus A321 NEO'},"{'name': 'British', 'iata': 'BA', 'icao': 'BAW'}"


In [24]:
df_departures = pd.DataFrame(result['departures_raw'])

In [25]:
df_departures

,movement,number,callSign,status,codeshareStatus,isCargo,aircraft,airline
0,"{'airport': {'icao': 'LEMD', 'iata': 'MAD', 'n...",IB 716,IBE07DC,Departed,IsOperator,False,"{'reg': 'EC-NTI', 'modeS': '34660B', 'model': ...","{'name': 'Iberia', 'iata': 'IB', 'icao': 'IBE'}"
1,"{'airport': {'icao': 'KSFO', 'iata': 'SFO', 'n...",VS 19,VIR19Z,Departed,IsOperator,False,"{'reg': 'G-VNYL', 'modeS': '407371', 'model': ...","{'name': 'Virgin Atlantic', 'iata': 'VS', 'ica..."
2,"{'airport': {'icao': 'LFMN', 'iata': 'NCE', 'n...",BA 344,BAW34GH,Departed,IsOperator,False,"{'reg': 'G-DBCJ', 'modeS': '400F99', 'model': ...","{'name': 'British', 'iata': 'BA', 'icao': 'BAW'}"
3,"{'airport': {'icao': 'KBOS', 'iata': 'BOS', 'n...",BA 213,BAW1B,Departed,IsOperator,False,"{'reg': 'G-YMMU', 'modeS': '405BFE', 'model': ...","{'name': 'British', 'iata': 'BA', 'icao': 'BAW'}"
4,"{'airport': {'icao': 'VTBS', 'iata': 'BKK', 'n...",TG 911,THA911,Departed,IsOperator,False,"{'reg': 'HS-TTA', 'modeS': '885281', 'model': ...","{'name': 'Thai International', 'iata': 'TG', ..."
...,...,...,...,...,...,...,...,...
594,"{'airport': {'icao': 'KDFW', 'iata': 'DFW', 'n...",AA 79,NaN,Expected,IsOperator,False,{'model': 'Boeing 787-9'},"{'name': 'American', 'iata': 'AA', 'icao': 'AAL'}"
595,"{'airport': {'icao': 'VOBL', 'iata': 'BLR', 'n...",VS 316,NaN,Expected,IsOperator,False,{'model': 'Boeing 787-9'},"{'name': 'Virgin Atlantic', 'iata': 'VS', 'ica..."
596,"{'airport': {'icao': 'EDDM', 'iata': 'MUC', 'n...",BA 948,NaN,Expected,IsOperator,False,{'model': 'Airbus A320'},"{'name': 'British', 'iata': 'BA', 'icao': 'BAW'}"
597,"{'airport': {'icao': 'LPPT', 'iata': 'LIS', 'n...",BA 506,NaN,Expected,IsOperator,False,{'model': 'Airbus A320 NEO'},"{'name': 'British', 'iata': 'BA', 'icao': 'BAW'}"


In [26]:
from datetime import datetime, timedelta

def calculate_imbalance(flight_data: dict) -> dict:
    """
    Calculate the cumulative imbalance metric from count_flights output.

    Splits flights into 30min buckets, computes per-bucket:
        100 * ((arrivals - departures) / max(arrivals + departures, 1))
    Then returns abs(sum of all bucket values).

    Args:
        flight_data: Return value from count_flights (must include arrivals_raw, departures_raw)

    Returns:
        dict with bucket_details, raw_sum, absolute_sum
    """
    start = datetime.strptime(flight_data["from"], "%Y-%m-%d %H:%M")
    end = datetime.strptime(flight_data["to"], "%Y-%m-%d %H:%M")

    def get_scheduled_local(f):
        movement = f.get("movement", {}) or {}
        st = (movement.get("scheduledTime", {}) or {})
        t = st.get("local") or st.get("utc")
        if t:
            return datetime.fromisoformat(t.replace("Z", "+00:00").replace("+00:00", ""))
        return None

    # ── Build 30min buckets ───────────────────────────────────────────────
    buckets = {}
    current = start
    while current < end:
        buckets[current] = {"arrivals": 0, "departures": 0}
        current += timedelta(minutes=30)

    def assign_to_bucket(flights, direction):
        for f in flights:
            t = get_scheduled_local(f)
            if t is None or t < start or t >= end:
                continue
            # Floor to nearest 30min bucket
            minutes_since_start = (t - start).total_seconds() / 60
            bucket_index = int(minutes_since_start // 30)
            bucket_key = start + timedelta(minutes=bucket_index * 30)
            if bucket_key in buckets:
                buckets[bucket_key][direction] += 1

    assign_to_bucket(flight_data["arrivals_raw"], "arrivals")
    assign_to_bucket(flight_data["departures_raw"], "departures")

    # ── Compute metric per bucket ─────────────────────────────────────────
    bucket_details = []
    running_sum = 0.0

    for t in sorted(buckets.keys()):
        a = buckets[t]["arrivals"]
        d = buckets[t]["departures"]
        metric = 100 * ((a - d) / max(a + d, 1))
        running_sum += metric
        bucket_details.append({
            "bucket": t.strftime("%Y-%m-%d %H:%M"),
            "arrivals": a,
            "departures": d,
            "metric": round(metric, 4),
        })

    return {
        "bucket_details": bucket_details,
        "raw_sum": round(running_sum, 4),
        "absolute_sum": round(abs(running_sum), 4),
    }

In [27]:
calculate_imbalance(result)

{'bucket_details': [{'bucket': '2026-02-28 12:00',
   'arrivals': 8,
   'departures': 21,
   'metric': -44.8276},
  {'bucket': '2026-02-28 12:30',
   'arrivals': 15,
   'departures': 20,
   'metric': -14.2857},
  {'bucket': '2026-02-28 13:00',
   'arrivals': 21,
   'departures': 21,
   'metric': 0.0},
  {'bucket': '2026-02-28 13:30',
   'arrivals': 18,
   'departures': 20,
   'metric': -5.2632},
  {'bucket': '2026-02-28 14:00',
   'arrivals': 16,
   'departures': 18,
   'metric': -5.8824},
  {'bucket': '2026-02-28 14:30',
   'arrivals': 22,
   'departures': 19,
   'metric': 7.3171},
  {'bucket': '2026-02-28 15:00',
   'arrivals': 19,
   'departures': 22,
   'metric': -7.3171},
  {'bucket': '2026-02-28 15:30',
   'arrivals': 20,
   'departures': 17,
   'metric': 8.1081},
  {'bucket': '2026-02-28 16:00',
   'arrivals': 16,
   'departures': 21,
   'metric': -13.5135},
  {'bucket': '2026-02-28 16:30',
   'arrivals': 15,
   'departures': 18,
   'metric': -9.0909},
  {'bucket': '2026-02-28 1